# First doing the imports

In [1]:
from whoosh.index import create_in
from whoosh.index import open_dir
from whoosh.fields import *
import json
import os
from tqdm import tqdm
from whoosh.query import *
from whoosh.qparser import QueryParser, FuzzyTermPlugin, OrGroup
import linecache
from google import genai
from pydantic import BaseModel
from typing import List
import pandas as pd
import numpy as np
import re

path_to_doc_chuncks = '/Users/tylergarfield/Desktop/school/CSC483/Rags_To_Searches/corpus/statpearls/chunk'

# Now we creat the index but we don't open it yet.

In [2]:
schema = Schema(doc_id = ID(stored=True), article_title = TEXT(stored=True), \
                section_title = TEXT(stored=True), content=TEXT(stored=True))

ix = create_in("woosh_index",schema)

# I want to look at the json chuncks first and perhaps do some prelinary cleanup getting the special sysmbols out of there.

In [3]:

docs_in_durr = os.listdir(path_to_doc_chuncks)

ind_writer = ix.writer()

try:

    for file in tqdm(docs_in_durr,total=len(docs_in_durr)):
    
        with open(f'{path_to_doc_chuncks}/{file}','r',encoding="utf-8") as f:
            
            for line in f:
                
                curr_art_chunk = json.loads(line)

                doc_id_curr = curr_art_chunk['id']
                cont_curr = curr_art_chunk['content']

                article_sec = curr_art_chunk['title'].split('--')
                main_title_curr = article_sec[0]
                sec_header_curr = article_sec[1]

                ind_writer.add_document(doc_id=doc_id_curr,article_title=main_title_curr,\
                                        section_title=sec_header_curr,content=cont_curr)

    ind_writer.commit()

    print("Woosh index was saved safely, let's look at this thing!")
        
except Exception as e:
    print("Owe no writing of index didn't complete successfully!❌")
    print(f"Error was: {e}")
    ind_writer.cancel()


ix.close()


100%|███████████████████████████████████████████████████████████████████████████████| 9515/9515 [01:09<00:00, 135.94it/s]


Woosh index was saved safely, let's look at this thing!


# Now that the index has been built I am going to test out querying it

In [5]:
def get_article_content(article_id):
    part_ids = article_id.split('_')
    
    main_article_id = part_ids[0]
    chunck_id = part_ids[1]

    line = linecache.getline(f'{path_to_doc_chuncks}/{main_article_id}.jsonl',(int(chunck_id)+1))
    art_json = json.loads(line)

    return art_json['content']

In [6]:
ix = open_dir("woosh_index")


with ix.searcher() as searcher:
    print(searcher.doc_count())
    parser = QueryParser("content",ix.schema)
    parser.add_plugin(FuzzyTermPlugin())

    #query_curr = parser.parse('section_title:(history AND physical)^10 AND ((headache AND backache)^0.2 OR diarhea~2^3.5 AND vomiting^2)')
    #query_curr = parser.parse('section_title:(history AND physical)^2 AND content:"headache vomiting"~1')
    query_curr = parser.parse('section_title:(history AND physical)^2 AND (content:headache~2^2 vomiting~2^2)')
    print(query_curr)
    print('\n\n')
    
    test_res = searcher.search(query_curr,limit=20)

    print("-------------Top Res Content------------")
    print(f'{test_res[0]["doc_id"]}: {get_article_content(test_res[0]["doc_id"])}\n\n')

    print("-------------Worst Res Content------------")
    print(f'{test_res[len(test_res.top_n)-1]["doc_id"]}: {get_article_content(test_res[len(test_res.top_n)-1]["doc_id"])}\n\n')


    print("-------------Possible Diseases------------")
    for res in test_res:
        print(f'\t {res.rank}) {res["article_title"]} |{res.score:.4f}|')

357680
(section_title:history^2.0 AND section_title:physical^2.0 AND content:headache~2^4.000000 AND content:vomiting~2^4.000000)



-------------Top Res Content------------
article-27743_23: The location of the tumor, supratentorial or infratentorial, is important for the manifestation of symptoms. Those with an infratentorial location usually develop hydrocephalus with headache, vomiting, irritability, and lethargy. Ataxia or other cerebellar signs and cranial nerve palsies are common. They rarely have seizures. In supratentorial locations, vomiting, seizures, and headaches are common. Hemiparesis is present if the tumor affects the cortical motor areas or the descending tracts. The presentation is also affected by age. Younger patients present irritability, vomiting, and visual problems. Those older than three years usually show headaches, vomiting, and ataxia. [8]


-------------Worst Res Content------------
article-25161_14: Patients usually present within a month of treatment wit

In [39]:
med_qs_path = '/Users/tylergarfield/Desktop/school/CSC483/Rags_To_Searches/50k.csv'
mqs_df = pd.read_csv(med_qs_path)
mqs_df['question'] = mqs_df['question'].apply(lambda x: re.sub(r'[^\w\s]','',test).lower())

In [31]:
ix = open_dir("woosh_index")



with ix.searcher() as searcher:

    parser = QueryParser("content",ix.schema)
    

    for record in mqs_df.head(5).to_dict('records'):

        print('---------------------------------------------------------')
        print(f"Question: {record['question']}")

        or_q = ' OR '.join(record['question'].split())

        query_curr = parser.parse(or_q)
        
        test_res = searcher.search(query_curr,limit=5)

        for res in test_res:
            print(f' {res.rank}) {res["article_title"]} |{res.score:.4f}|')

            print(f'\t\t {get_article_content(res["doc_id"])}')
ix.close()

---------------------------------------------------------
Question: best confirmation for pregnancy at six weeks
 0) Physiology, Pregnancy  |27.7661|
		 Confirmation of a viable pregnancy occurs with an ultrasound, which shows a gestational sac on transvaginal ultrasound at five weeks or with an hCG level of 1,500 to 2,000 mIU/mL. Fetal heart motion is visible on transvaginal ultrasound at six weeks or with hCG levels starting at 5,000 to 6,000 mIU/mL. [12]
 1) Schistosomiasis  |21.1507|
		 Praziquantel in these dosages effectively kills adult worms and prevents the release of eggs and the development of new urogenital lesions.  As it is primarily effective against adult worms, treatment is best initiated at least four to six weeks post-exposure. [23] [24] [25] While some resistance has been noted in other species of Schistosoma, it has been minimal in S. haematobium . Praziquantel is generally safe in pregnancy, rated category B.
 2) Endocarditis Antibiotic Regimens  |21.0468|
		 MSSA

In [3]:
ix = open_dir("woosh_index")

med_qs_path = '/Users/tylergarfield/Desktop/school/CSC483/Rags_To_Searches/50k.csv'
mqs_df = pd.read_csv(med_qs_path)
opc_cols = ['opa','opb','opc','opd']
failed_queries = []
zero_div_qs = []

mrr = 0
with ix.searcher() as searcher:

    parser = QueryParser("content",ix.schema,group=OrGroup.factory())
    
    benchmark_qs = mqs_df.to_dict('records')
    tot_mr = 0
    pbar = tqdm(benchmark_qs,total=len(benchmark_qs))
    pbar.set_description('Zero failed questions')
    for record in pbar:

        try:
            correct_ans_colind = record['cop']
            correct_ans = record[opc_cols[correct_ans_colind]].lower().strip()
    
            query_curr = parser.parse(record['question'])
            
            
            test_res = searcher.search(query_curr,limit=15)
    
            rank = 0
            for res in test_res:
                rank_j = res.pos+1
                
    
                if correct_ans in res["content"].lower():
                    rank = rank_j
                    break
    
            if rank != 0:
                tot_mr += 1/rank
            else:
                failed_queries.append(record['question'])
                pbar.set_description(f'Failed zero res queries at {len(failed_queries)} Zero div {len(zero_div_qs)}')
                
        except ZeroDivisionError:
            zero_div_qs.append(record['question'])
            pbar.set_description(f'Failed zero res queries at {len(failed_queries)} Zero div {len(zero_div_qs)}')

    mrr = tot_mr/len(benchmark_qs)
            
ix.close()

print(f'MRR at 15 is: {mrr}') 

Failed zero res queries at 255 Zero div 42:   1%|▎                                 | 388/50000 [02:19<4:56:27,  2.79it/s]


KeyboardInterrupt: 